# Pro-Cure · Notebook 01 · OCDS Schema Exploration & Parser

**Monday 26 May 2026**

### Goals
1. Understand the exact column structure of the SA National Treasury OCDS CSV
2. Document all known data quality issues
3. Build a robust parser that loads and cleans all years into one master DataFrame
4. Filter for health sector procurement only
5. Export `health_sector_master.csv`

### Data source
National Treasury eTender OCDS · [data.etenders.gov.za](https://data.etenders.gov.za)  
Licence: PDDL (fully open) · Format: CSV, one file per month

---
> *"2,207 procurement bundles. That is what Babita Deokaran read manually.  
> This notebook reads them algorithmically."*


## 0. Imports & paths

In [ ]:
import os
import glob
import warnings
import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")

# Paths
RAW_DIR       = "../data/raw"
PROCESSED_DIR = "../data/processed"
HEALTH_DIR    = "../data/health_sector"

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(HEALTH_DIR,    exist_ok=True)

print("✅ Imports OK")
print(f"   Raw data dir:       {os.path.abspath(RAW_DIR)}")
print(f"   Processed data dir: {os.path.abspath(PROCESSED_DIR)}")
print(f"   Health sector dir:  {os.path.abspath(HEALTH_DIR)}")


✅ Imports OK
   Raw data dir:       /Users/kamogelomotlhale/Documents/Passion Projects/pro-cure/pro-cure/data/raw
   Processed data dir: /Users/kamogelomotlhale/Documents/Passion Projects/pro-cure/pro-cure/data/processed
   Health sector dir:  /Users/kamogelomotlhale/Documents/Passion Projects/pro-cure/pro-cure/data/health_sector


## 1. OCDS column schema reference

These are the **exact** column names in the SA National Treasury OCDS CSV files,  
confirmed from a live download in May 2026.  

Source: `data.etenders.gov.za/Home/DownloadFile/?fileName=01012025.csv`


In [ ]:
# Full column schema — confirmed from live SA OCDS CSV
OCDS_COLUMNS = [
    # Identity
    "ocid",                            # Unique contracting process ID (ocds-9t57fa-XXXXXX)
    "id",                              # Release ID (ocid + date)
    "date",                            # Release date — format: YYYY/MM/DD HH:MM:SS
    "tag",                             # ⚠️  BROKEN — contains .NET object strings, not usable
    "initiationtype",                  # Always 'tender' in SA data

    # Buyer
    "buyer_name",                      # ⭐ Government entity name — KEY FIELD
    "buyer_id",                        # Numeric entity ID
    "publisher_name",                  # Always 'National Treasury (South Africa)'
    "license",                         # PDDL licence URL

    # Planning
    "planning_rationale",              # Why procurement was initiated (often empty)
    "planning_budget_description",     # Budget line description (often ' - ')

    # Tender
    "tender_id",                       # ⭐ Tender number — KEY FIELD (splitting detection)
    "tender_title",                    # Short title
    "tender_description",              # ⭐ Full description — KEY FIELD for NLP
    "tender_status",                   # active / complete / cancelled
    "tender_classification",           # ⭐ Procurement category — KEY FIELD (price anomaly)
    "tender_period_startdate",
    "tender_period_enddate",
    "tender_eligibility_criteria",
    "tender_contractperiod_startdate",
    "tender_contractperiod_enddate",
    "tender_document_urls",

    # Bidders
    "bidders",                         # ⚠️  Almost always empty in SA data

    # Awards
    "awards_id",
    "award_status",                    # active / unsuccessful
    "awards_date",                     # ⭐ Award date — KEY FIELD (temporal clustering)
    "awards_suppliers_name",           # ⭐ Winning supplier name — KEY FIELD

    # Contracts
    "contracts_id",
    "contracts_title",
    "contract_description",
    "contracts_period_startdate",
    "contracts_period_enddate",
    "contracts_status",

    # Milestones
    "contracts_milestones_id",
    "contracts_milestones_title",
    "contracts_milestones_type",
    "contracts_milestones_description",
    "contracts_milestones_duedate",
]

# Pro-Cure key fields
KEY_FIELDS = {
    "buyer_name":           "Which entity is buying — hospital, department, etc.",
    "tender_description":   "Text for NLP pipeline — single-source detection",
    "tender_id":            "Contract splitting — same ID, split values",
    "tender_classification":"Commodity category for price anomaly grouping",
    "awards_date":          "Temporal clustering for splitting detection",
    "awards_suppliers_name":"Supplier identity — network and concentration analysis",
    "award_status":         "Filter for active awards only",
    "tender_status":        "Filter complete/cancelled tenders",
}

print(f"Total columns in SA OCDS CSV : {len(OCDS_COLUMNS)}")
print(f"Key fields for Pro-Cure      : {len(KEY_FIELDS)}")
print()
print(f"{'Field':<40} Description")
print("-" * 75)
for field, desc in KEY_FIELDS.items():
    print(f"  ⭐ {field:<37} {desc}")


## 2. Documented data quality issues

These are **confirmed issues** from the OCP Data Registry entry for SA National Treasury (April 2026).  
The parser below handles all of them automatically.


In [ ]:
DATA_QUALITY_ISSUES = {
    "duplicate_releases": (
        "A few duplicate releases with identical release IDs and content.\n"
        "Fix: deduplicate on 'ocid' + 'date' after loading."
    ),
    "empty_arrays_and_placeholders": (
        "Data includes empty fields, empty arrays, and invalid values including ' - '.\n"
        "Fix: replace all placeholder strings with NaN."
    ),
    "missing_parties_array": (
        "The parties array is missing for a majority of contracting processes.\n"
        "Fix: use buyer_name + awards_suppliers_name as primary identifiers instead."
    ),
    "broken_tag_field": (
        "The 'tag' field contains serialised .NET list objects:\n"
        "  'System.Collections.Generic.List`1[System.String]' — completely unusable.\n"
        "Fix: ignore tag field, use tender_status and award_status instead."
    ),
    "non_iso_date_format": (
        "Dates formatted as 'YYYY/MM/DD HH:MM:SS' — not ISO 8601.\n"
        "Fix: parse with pd.to_datetime(format='%Y/%m/%d %H:%M:%S')."
    ),
    "no_value_in_csv": (
        "⚠️  CONTRACT VALUE is NOT in the CSV schema.\n"
        "tender_value_amount only exists in the JSON format.\n"
        "Fix: download JSON files alongside CSV for value-dependent analysis."
    ),
}

print("Documented data quality issues:\n")
for issue, explanation in DATA_QUALITY_ISSUES.items():
    print(f"  ⚠️  {issue}")
    for line in explanation.split("\n"):
        print(f"      {line}")
    print()


## 3. Health sector keywords

In [ ]:
HEALTH_KEYWORDS = [
    # National and provincial departments
    "department of health", "dept of health", "health department",
    "gauteng department of health", "kwazulu-natal department of health",
    "kzn department of health", "limpopo department of health",
    "mpumalanga department of health", "eastern cape department of health",
    "western cape department of health", "northern cape department of health",
    "north west department of health", "free state department of health",
    "national department of health",

    # Hospitals
    "hospital", "tembisa hospital", "chris hani baragwanath",
    "charlotte maxeke", "helen joseph", "groote schuur",
    "tygerberg", "inkosi albert luthuli", "pelonomi", "mankweng",

    # Other health entities
    "clinic", "health centre", "health center",
    "district health", "provincial health",
    "health authority", "health services", "medical supplies",
]

print(f"Health sector keywords: {len(HEALTH_KEYWORDS)}")
print("\nSample keywords:")
for kw in HEALTH_KEYWORDS[:8]:
    print(f"  · {kw}")
print("  ...")


## 4. Parser functions

In [ ]:
def load_ocds_csv(filepath):
    """Load a single OCDS CSV — handles encoding quirks."""
    try:
        df = pd.read_csv(filepath, encoding="utf-8", low_memory=False, on_bad_lines="skip")
    except UnicodeDecodeError:
        df = pd.read_csv(filepath, encoding="latin-1", low_memory=False, on_bad_lines="skip")
    return df


def clean_ocds_df(df):
    """Apply all documented data quality fixes."""
    original_rows = len(df)

    # Fix 1: Replace known placeholder values with NaN
    PLACEHOLDERS = [" - ", "-", "N/A", "n/a", "None", "none", "NULL", "null", ""]
    df = df.replace(PLACEHOLDERS, np.nan)

    # Fix 2: Neutralise broken tag field
    if "tag" in df.columns:
        df["tag"] = np.nan

    # Fix 3: Parse dates
    date_cols = [c for c in df.columns if "date" in c.lower()]
    for col in date_cols:
        df[col] = pd.to_datetime(df[col], format="%Y/%m/%d %H:%M:%S", errors="coerce")

    # Fix 4: Deduplicate
    if "ocid" in df.columns and "date" in df.columns:
        df = df.drop_duplicates(subset=["ocid", "date"], keep="first")

    # Fix 5: Normalise buyer and supplier names to UPPERCASE
    for col in ["buyer_name", "awards_suppliers_name"]:
        if col in df.columns:
            df[col] = df[col].str.strip().str.upper()

    # Fix 6: Add year column
    if "date" in df.columns:
        df["year"] = pd.to_datetime(df["date"], errors="coerce").dt.year

    cleaned_rows = len(df)
    dropped = original_rows - cleaned_rows
    print(f"    {original_rows:>8,} → {cleaned_rows:>8,} rows  ({dropped:,} duplicates removed)")
    return df


def load_all_years(raw_dir):
    """Load and concatenate all OCDS CSV files from raw_dir."""
    all_files = sorted(glob.glob(os.path.join(raw_dir, "*.csv")))

    if not all_files:
        print(f"⚠️  No CSV files found in {raw_dir}/")
        print("   Download from: data.open-contracting.org/en/publication/143")
        print("   Place .csv files in data/raw/ and re-run this cell.")
        return pd.DataFrame()

    print(f"Found {len(all_files)} CSV file(s)\n")
    print(f"  {'File':<30} {'Raw rows':>10}  {'After clean':>12}")
    print("  " + "-" * 56)

    dfs = []
    for fpath in all_files:
        fname = os.path.basename(fpath)
        print(f"  {fname:<30}", end="")
        df = load_ocds_csv(fpath)
        print(f" {len(df):>10,} ", end="")
        df = clean_ocds_df(df)
        dfs.append(df)

    master = pd.concat(dfs, ignore_index=True)
    print(f"\n✅ Master DataFrame: {len(master):,} rows · {len(master.columns)} columns")
    return master


def filter_health_sector(df):
    """Filter to health sector records using keyword match on buyer_name."""
    if df.empty:
        return df
    pattern = "|".join(HEALTH_KEYWORDS)
    mask = df["buyer_name"].str.lower().str.contains(pattern, na=False, regex=True)
    health_df = df[mask].copy()
    print(f"Health sector filter:  {len(df):,} total → {len(health_df):,} health records ({100*len(health_df)/len(df):.1f}%)")
    return health_df


def null_rate_report(df, label="DataFrame"):
    """Print null rate for every column."""
    print(f"\nNull rate report — {label} ({len(df):,} rows)\n")
    print(f"  {'Column':<45} {'Non-null':>10}  {'Null%':>7}")
    print("  " + "-" * 67)
    for col in df.columns:
        non_null = df[col].notna().sum()
        null_pct = 100 * (1 - non_null / len(df))
        flag = "  ⚠️  mostly empty" if null_pct > 80 else ""
        print(f"  {col:<45} {non_null:>10,}  {null_pct:>6.1f}%{flag}")


print("✅ Parser functions defined — ready to run")


## 5. Load all years

Run this cell after placing your downloaded CSV files in `data/raw/`.


In [ ]:
master_df = load_all_years(RAW_DIR)


## 6. Schema & null rate report

In [ ]:
if not master_df.empty:
    null_rate_report(master_df, label="Full master dataset")


## 7. Save master dataset

In [ ]:
if not master_df.empty:
    master_path = os.path.join(PROCESSED_DIR, "master.csv")
    master_df.to_csv(master_path, index=False)
    print(f"✅ Saved: {master_path}  ({len(master_df):,} rows)")


## 8. Health sector filter

In [ ]:
if not master_df.empty:
    health_df = filter_health_sector(master_df)

    print(f"\nTop 20 health sector buyers:")
    print("-" * 55)
    top = health_df["buyer_name"].value_counts().head(20)
    for buyer, count in top.items():
        tembisa = " ← 🎯 TEMBISA" if "TEMBISA" in buyer else ""
        print(f"  {count:>6,}  {buyer}{tembisa}")


## 9. Null rate report — health sector

In [ ]:
if not master_df.empty and not health_df.empty:
    null_rate_report(health_df, label="Health sector subset")


## 10. Save health sector dataset

In [ ]:
if not master_df.empty and not health_df.empty:
    health_path = os.path.join(HEALTH_DIR, "health_sector_master.csv")
    health_df.to_csv(health_path, index=False)
    print(f"✅ Saved: {health_path}  ({len(health_df):,} rows)")
    print()
    print("Day 2 complete.")
    print("Next → notebooks/02_eda.ipynb — first corruption signals in the data")


## ⚠️ Important: download JSON files too

The CSV format **does not include contract values** (`tender_value_amount`).  
Contract values are only available in the **JSON format**.

**Also download** from `data.open-contracting.org/en/publication/143`:
- `2021.json` — Tembisa peak year
- `2022.json`
- `2023.json`

Place them in `data/raw/`. Notebook 03 (contract splitting) will parse these for values.

---

**Commit when done:**
```bash
git add notebooks/01_data_parsing.ipynb
git commit -m "feat: Day 2 — OCDS parser notebook with schema, quality fixes, health sector filter"
git push
```
